# 1 · Sources — one wrapper over every input

`digitalearth.sources` turns any pyramids object (or a raw numpy array) into a uniform `Source` of `(z, x, y, crs, metadata)`. It is the data layer every plot method reads from. pyramids is the only GIS dependency — no xarray/rasterio.

In [1]:
%matplotlib inline
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs
# from docs/examples/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "acc4000.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RASTER = str(ROOT / "examples" / "data" / "acc4000.tif")
POINTS = str(ROOT / "tests" / "data" / "points.geojson")
print("data root:", ROOT)

data root: C:\gdrive\algorithms\Visualization\Digital-Earth


## From a pyramids `Dataset` (raster)

In [2]:
from pyramids.dataset import Dataset
from digitalearth.sources import get_source

ds = Dataset.read_file(RASTER)
src = get_source(ds)
print('kind     :', src.metadata('kind'))
print('variable :', src.metadata('variable'))
print('z shape  :', src.z.values.shape)
print('x / y len:', src.x.values.shape, src.y.values.shape)
print('crs      :', src.crs)

2026-05-28 00:30:34 | INFO | pyramids.base.config | Logging is configured.


kind     : raster
variable : Band_1
z shape  : (13, 14)
x / y len: (14,) (13,)
crs      : 32618


The data dimension is NaN-masked at the nodata value:

In [3]:
import numpy as np
print('has NaN (masked nodata):', bool(np.isnan(src.z.values).any()))

has NaN (masked nodata): True


## From a raw numpy array

No CRS; pixel-index axes unless you pass `x`/`y`.

In [4]:
arr = np.arange(12.0).reshape(3, 4)
s = get_source(arr)
print('x:', s.x.values.tolist())
print('crs:', s.crs)

x: [0.0, 1.0, 2.0, 3.0]
crs: None


## From a `FeatureCollection` (points)

`FeatureCollection` is a GeoDataFrame subclass; the first numeric column becomes `z`.

In [5]:
from pyramids.feature import FeatureCollection

fc = FeatureCollection.read_file(POINTS)
sv = get_source(fc)
print('kind :', sv.metadata('kind'))
print('npts :', sv.x.values.shape[0])
print('crs  :', sv.crs)

kind : vector
npts : 10
crs  : 32618


## From a `DatasetCollection` (a member)

In [6]:
from pyramids.dataset.collection import DatasetCollection

dc = DatasetCollection.from_files([RASTER, RASTER])
sc = get_source(dc)
print('member / n_members:', sc.metadata('member'), '/', sc.metadata('n_members'))

member / n_members: 0 / 2
